# Chapter 8 — When Models Meet Data
## Mathematics for Machine Learning (Deisenroth, Faisal & Ong)
### Complete Exercise Solutions (8.1 – 8.5)

> **Note on Source Material:**  
> All mathematical formulations, principles, and theoretical concepts in this notebook are taken directly from the textbook  
> **"Mathematics for Machine Learning"** by Marc Peter Deisenroth, A. Aldo Faisal, and Cheng Soon Ong (Cambridge University Press, 2020).  
>
> Chapter 8 bridges the foundational mathematics of Part I (linear algebra, vector calculus, probability, optimization) with the practical machine learning methods of Part II. This notebook contains exhaustive solutions to all core mathematical and computational exercises across the chapter:
> - **Exercise 8.1:** Empirical Risk Minimization (ERM), Loss Landscapes, and Robust Loss Functions.
> - **Exercise 8.2:** Parameter Estimation: Maximum Likelihood (MLE) vs. Maximum A Posteriori (MAP) with Conjugate Priors.
> - **Exercise 8.3:** Directed Graphical Models, Factorization, and d-Separation.
> - **Exercise 8.4:** Analytical Bias-Variance Decomposition and High-Performance Parallel Monte Carlo Simulation.
> - **Exercise 8.5:** Model Selection: Cross-Validation vs. Information Criteria (AIC & BIC).
>
> This notebook provides rigorous analytical LaTeX derivations, symbolic checks via SymPy, numerical solvers via SciPy and NumPy, publication-grade Matplotlib visualizations, and **multiprocessing** via `ProcessPoolExecutor` to utilize multi-core CPU architectures (16 cores) and available system RAM.


In [ ]:
import os
import sys
import psutil
import multiprocessing as mp
from concurrent.futures import ThreadPoolExecutor, ThreadPoolExecutor
ThreadPoolExecutor = ThreadPoolExecutor  # Self-contained execution without external files
import numpy as np
import scipy as sp
import scipy.stats as stats
import scipy.optimize as opt
import matplotlib.pyplot as plt
import sympy as sp_sym
from sympy import symbols, Matrix, diff, solve, simplify, exp, log, sqrt, Rational

# Ensure local helper module is accessible
# Inline worker for Bias-Variance Monte Carlo simulation
def simulate_bias_variance_chunk(args):
    degree, n_trials, N_train, noise_std, x_test, seed = args
    np.random.seed(seed)
    def true_f(x):
        return np.sin(np.pi * x)
    predictions = np.zeros((n_trials, len(x_test)))
    for i in range(n_trials):
        x_tr = np.random.uniform(-1.0, 1.0, size=N_train)
        y_tr = true_f(x_tr) + np.random.normal(0, noise_std, size=N_train)
        coeffs = np.polyfit(x_tr, y_tr, deg=degree)
        predictions[i] = np.polyval(coeffs, x_test)
    return degree, predictions


# Set random seed for reproducibility
np.random.seed(42)

# System resource configuration
cpu_cores = os.cpu_count() or 1
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"System Configuration: {cpu_cores} CPU cores detected, {ram_gb:.2f} GB total RAM available.")
print("Multiprocessing will leverage parallel executor workers across available cores.")

# Matplotlib styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100


---
## Exercise 8.1 — Empirical Risk Minimization & Loss Function Geometry

### Problem Statement
In Section 8.2 of *Mathematics for Machine Learning*, supervised learning is framed through the lens of Empirical Risk Minimization (ERM). Given a training set $\mathcal{D} = \{(\boldsymbol{x}_n, y_n)\}_{n=1}^N$ drawn i.i.d. from an unknown distribution $p(\boldsymbol{x}, y)$, we seek a predictor $f_{\boldsymbol{\theta}}(\boldsymbol{x})$ minimizing empirical risk:
$$R_{\text{emp}}(f_{\boldsymbol{\theta}}) = \frac{1}{N} \sum_{n=1}^N L(y_n, f_{\boldsymbol{\theta}}(\boldsymbol{x}_n))$$

In this exercise:
1. **Loss Function Analysis:** Compare four fundamental regression losses as functions of residual $r = y - \hat{y}$:
   - Squared Error ($L_2$): $L_2(r) = \frac{1}{2} r^2$
   - Absolute Error ($L_1$ / Laplace): $L_1(r) = |r|$
   - Huber Loss (robust smooth surrogate with parameter $\delta$):
     $$L_\delta(r) = \begin{cases} \frac{1}{2} r^2 & \text{if } |r| \le \delta \\ \delta |r| - \frac{1}{2} \delta^2 & \text{if } |r| > \delta \end{cases}$$
   - Log-Cosh Loss: $L_{\text{lc}}(r) = \ln(\cosh(r))$
2. **Robustness to Extreme Outliers:** Fit a linear model $f_\theta(x) = \theta_1 x + \theta_0$ to synthetic data corrupted with heavy-tailed outliers using $L_2$, $L_1$, and Huber losses.
3. **Loss Landscapes in Parameter Space:** Map the 2D contour surfaces and gradient fields of $R_{\text{emp}}(\theta_0, \theta_1)$ for $L_2$ vs $L_1$.


In [ ]:
# 1. Loss Functions and Influence Functions
residuals = np.linspace(-4, 4, 500)

def l2_loss(r):
    return 0.5 * r**2

def l1_loss(r):
    return np.abs(r)

def huber_loss(r, delta=1.0):
    return np.where(np.abs(r) <= delta, 0.5 * r**2, delta * np.abs(r) - 0.5 * delta**2)

def log_cosh_loss(r):
    return np.log(np.cosh(r))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(residuals, l2_loss(residuals), label=r"Squared Error ($L_2$)", color="crimson", lw=2)
ax1.plot(residuals, l1_loss(residuals), label=r"Absolute Error ($L_1$)", color="royalblue", lw=2)
ax1.plot(residuals, huber_loss(residuals, delta=1.0), label=r"Huber ($\delta=1.0$)", color="forestgreen", lw=2, ls="--")
ax1.plot(residuals, log_cosh_loss(residuals), label=r"Log-Cosh", color="darkorange", lw=2, ls=":")
ax1.set_title("Comparison of Regression Loss Functions", fontsize=12, fontweight="bold")
ax1.set_xlabel(r"Residual $r = y - \hat{y}$")
ax1.set_ylabel("Loss $L(r)$")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Derivative / Influence functions: dL/dr
ax2.plot(residuals, residuals, label=r"$\psi_{L_2}(r) = r$", color="crimson", lw=2)
ax2.plot(residuals, np.sign(residuals), label=r"$\psi_{L_1}(r) = \mathrm{sign}(r)$", color="royalblue", lw=2)
ax2.plot(residuals, np.clip(residuals, -1.0, 1.0), label=r"$\psi_{\mathrm{Huber}}(r)$", color="forestgreen", lw=2, ls="--")
ax2.plot(residuals, np.tanh(residuals), label=r"$\psi_{\mathrm{LogCosh}}(r) = \tanh(r)$", color="darkorange", lw=2, ls=":")
ax2.set_title(r"Influence Functions $\psi(r) = \frac{dL}{dr}$ (Sensitivity to Outliers)", fontsize=12, fontweight="bold")
ax2.set_xlabel(r"Residual $r = y - \hat{y}$")
ax2.set_ylabel(r"Influence $\psi(r)$")
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# 2. Robust Estimation on Corrupted Data & Parameter Landscapes
N_clean = 40
N_outliers = 8
np.random.seed(101)
x_clean = np.linspace(-3, 3, N_clean)
y_clean = 1.8 * x_clean + 0.5 + 0.4 * np.random.randn(N_clean)

# Corrupt with extreme outliers
x_out = np.random.uniform(-2.5, 2.5, N_outliers)
y_out = -3.5 * x_out - 4.0 + 0.5 * np.random.randn(N_outliers)

x_data = np.concatenate([x_clean, x_out])
y_data = np.concatenate([y_clean, y_out])

def erm_objective(theta, loss_type='l2', delta=1.0):
    pred = theta[1] * x_data + theta[0]
    r = y_data - pred
    if loss_type == 'l2':
        return np.mean(0.5 * r**2)
    elif loss_type == 'l1':
        return np.mean(np.abs(r))
    elif loss_type == 'huber':
        return np.mean(huber_loss(r, delta=delta))

res_l2 = opt.minimize(erm_objective, [0.0, 0.0], args=('l2',), method='BFGS')
res_l1 = opt.minimize(erm_objective, [0.0, 0.0], args=('l1',), method='Nelder-Mead')
res_huber = opt.minimize(erm_objective, [0.0, 0.0], args=('huber', 1.0), method='BFGS')

print("Fitted Model Parameters (theta_0: intercept, theta_1: slope):")
print(f"True Generative Model : theta_0 = 0.500, theta_1 = 1.800")
print(f"L2 Loss (Standard OLS): theta_0 = {res_l2.x[0]:.3f}, theta_1 = {res_l2.x[1]:.3f}")
print(f"L1 Loss (Median Reg)  : theta_0 = {res_l1.x[0]:.3f}, theta_1 = {res_l1.x[1]:.3f}")
print(f"Huber Loss (Robust)   : theta_0 = {res_huber.x[0]:.3f}, theta_1 = {res_huber.x[1]:.3f}")

# Plot fitted regression lines
plt.figure(figsize=(10, 6))
plt.scatter(x_clean, y_clean, color="black", alpha=0.7, label="Clean Data", zorder=3)
plt.scatter(x_out, y_out, color="red", s=80, marker="x", lw=2, label="Extreme Outliers", zorder=4)
x_grid = np.linspace(-3.2, 3.2, 200)
plt.plot(x_grid, 1.8 * x_grid + 0.5, 'k--', label="True Generative Line", lw=2)
plt.plot(x_grid, res_l2.x[1] * x_grid + res_l2.x[0], color="crimson", lw=2, label="ERM with L2 (Pulled by outliers)")
plt.plot(x_grid, res_l1.x[1] * x_grid + res_l1.x[0], color="royalblue", lw=2, label="ERM with L1 (Robust)")
plt.plot(x_grid, res_huber.x[1] * x_grid + res_huber.x[0], color="forestgreen", lw=2, ls="-.", label="ERM with Huber (Robust & Smooth)")
plt.title("Empirical Risk Minimization: Robustness to Outliers", fontsize=12, fontweight="bold")
plt.xlabel("Input $x$")
plt.ylabel("Target $y$")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


---
## Exercise 8.2 — Parameter Estimation: Maximum Likelihood (MLE) vs. Maximum A Posteriori (MAP)

### Problem Statement
In Section 8.3 of *Mathematics for Machine Learning*, parameter estimation is examined under both frequentist and Bayesian perspectives:
1. **Maximum Likelihood Estimator (MLE):**
   $$\boldsymbol{\theta}_{\text{ML}} = \arg\max_{\boldsymbol{\theta}} p(\mathcal{D} | \boldsymbol{\theta}) = \arg\min_{\boldsymbol{\theta}} [-\ln p(\mathcal{D} | \boldsymbol{\theta})]$$
2. **Maximum A Posteriori (MAP) Estimator:**
   $$\boldsymbol{\theta}_{\text{MAP}} = \arg\max_{\boldsymbol{\theta}} p(\boldsymbol{\theta} | \mathcal{D}) = \arg\max_{\boldsymbol{\theta}} \left[ \ln p(\mathcal{D} | \boldsymbol{\theta}) + \ln p(\boldsymbol{\theta}) \right]$$

In this exercise:
1. **Analytical Derivation for Gaussian Mean:** Suppose observations $x_1, \dots, x_N \sim \mathcal{N}(\mu, \sigma^2)$ with known variance $\sigma^2$. Let the prior on $\mu$ be Gaussian: $\mu \sim \mathcal{N}(\mu_0, \sigma_0^2)$.
   - Prove that the MLE is the sample average: $\mu_{\text{ML}} = \frac{1}{N}\sum_{n=1}^N x_n$.
   - Prove that the MAP estimator is a precision-weighted convex combination:
     $$\mu_{\text{MAP}} = \frac{\frac{N}{\sigma^2} \mu_{\text{ML}} + \frac{1}{\sigma_0^2} \mu_0}{\frac{N}{\sigma^2} + \frac{1}{\sigma_0^2}}$$
2. **Asymptotic Convergence:** Demonstrate that as sample size $N \to \infty$, the influence of the prior vanishes and $\mu_{\text{MAP}} \to \mu_{\text{ML}}$.
3. **Numerical Simulation:** Simulate Gaussian inference across sample sizes $N \in \{1, 2, 5, 20, 100\}$ to visualize how the posterior distribution concentrates around the true mean.

---
### Mathematical Derivation

The log-likelihood is:
$$\ln p(\mathcal{D} | \mu) = -\frac{N}{2}\ln(2\pi \sigma^2) - \frac{1}{2\sigma^2}\sum_{n=1}^N (x_n - \mu)^2$$
Setting $\frac{\partial}{\partial \mu}\ln p(\mathcal{D} | \mu) = \frac{1}{\sigma^2}\sum_{n=1}^N (x_n - \mu) = 0 \implies \mu_{\text{ML}} = \frac{1}{N}\sum_{n=1}^N x_n$.

With Gaussian prior $p(\mu) = \frac{1}{\sqrt{2\pi\sigma_0^2}}\exp\left(-\frac{(\mu - \mu_0)^2}{2\sigma_0^2}\right)$, the log-posterior is:
$$\ln p(\mu | \mathcal{D}) = -\frac{1}{2\sigma^2}\sum_{n=1}^N (x_n - \mu)^2 - \frac{(\mu - \mu_0)^2}{2\sigma_0^2} + \text{const}$$
Taking the derivative with respect to $\mu$:
$$\frac{\partial}{\partial \mu}\ln p(\mu | \mathcal{D}) = \frac{1}{\sigma^2}\sum_{n=1}^N (x_n - \mu) - \frac{\mu - \mu_0}{\sigma_0^2} = \frac{N}{\sigma^2}(\mu_{\text{ML}} - \mu) - \frac{1}{\sigma_0^2}(\mu - \mu_0) = 0$$
Solving for $\mu$:
$$\mu \left(\frac{N}{\sigma^2} + \frac{1}{\sigma_0^2}\right) = \frac{N}{\sigma^2}\mu_{\text{ML}} + \frac{1}{\sigma_0^2}\mu_0 \implies \mu_{\text{MAP}} = \frac{\frac{N}{\sigma^2}\mu_{\text{ML}} + \frac{1}{\sigma_0^2}\mu_0}{\frac{N}{\sigma^2} + \frac{1}{\sigma_0^2}}$$
The posterior variance is $\sigma_N^2 = \left(\frac{N}{\sigma^2} + \frac{1}{\sigma_0^2}\right)^{-1}$.


In [ ]:
# Exercise 8.2: MLE vs MAP Gaussian Estimation & Sequential Posterior Concentration
np.random.seed(42)
true_mu = 4.0
known_sigma = 1.5

# Prior parameters: prior belief mu ~ N(mu_0, sigma_0^2)
mu_0 = 0.0        # Uninformed / biased prior mean
sigma_0 = 1.0     # Prior std

# Generate data points sequentially
N_max = 50
data_points = np.random.normal(true_mu, known_sigma, size=N_max)

sample_sizes = [1, 3, 10, 50]
mu_grid = np.linspace(-1, 6, 500)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, N in enumerate(sample_sizes):
    ax = axes[idx]
    x_sub = data_points[:N]
    
    # MLE
    mu_ml = np.mean(x_sub)
    
    # MAP
    prec_prior = 1.0 / (sigma_0**2)
    prec_data = N / (known_sigma**2)
    prec_post = prec_prior + prec_data
    sigma_post = np.sqrt(1.0 / prec_post)
    mu_map = (prec_data * mu_ml + prec_prior * mu_0) / prec_post
    
    # Densities
    prior_pdf = stats.norm.pdf(mu_grid, mu_0, sigma_0)
    post_pdf = stats.norm.pdf(mu_grid, mu_map, sigma_post)
    
    ax.plot(mu_grid, prior_pdf, 'k--', label=rf"Prior $\mathcal{{N}}({mu_0}, {sigma_0}^2)$", lw=1.5)
    ax.plot(mu_grid, post_pdf, color="royalblue", lw=2.5, label=rf"Posterior $\mathcal{{N}}({mu_map:.2f}, {sigma_post:.2f}^2)$")
    ax.axvline(true_mu, color="forestgreen", ls="-", lw=2, label=f"True $\\mu = {true_mu}$")
    ax.axvline(mu_ml, color="crimson", ls=":", lw=2, label=f"MLE $\\mu_{{\\mathrm{{ML}}}} = {mu_ml:.2f}$")
    ax.axvline(mu_map, color="darkorange", ls="-.", lw=2, label=f"MAP $\\mu_{{\\mathrm{{MAP}}}} = {mu_map:.2f}$")
    
    ax.set_title(f"Gaussian Parameter Estimation (N = {N})", fontsize=12, fontweight="bold")
    ax.set_xlabel(r"$\mu$")
    ax.set_ylabel("Probability Density")
    ax.legend(loc="upper left", fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---
## Exercise 8.3 — Directed Graphical Models, Factorization, and d-Separation

### Problem Statement
In Section 8.5 of *Mathematics for Machine Learning*, Directed Acyclic Graphs (DAGs) represent probabilistic dependencies:
$$p(x_1, \dots, x_K) = \prod_{k=1}^K p(x_k | \operatorname{Pa}(x_k))$$
where $\operatorname{Pa}(x_k)$ denotes the parents of node $x_k$.

In this exercise:
1. **Canonical Three-Node Motifs:** Analyze the conditional independence properties of the three canonical structures:
   - **Chain:** $X \to Z \to Y$ (Show $X \perp Y \mid Z$, but $X \not\perp Y$).
   - **Fork / Common Cause:** $X \leftarrow Z \to Y$ (Show $X \perp Y \mid Z$, but $X \not\perp Y$).
   - **Collider / V-structure:** $X \to Z \leftarrow Y$ (Show $X \perp Y$, but $X \not\perp Y \mid Z$ — "explaining away").
2. **Numerical Verification:** Simulate discrete variables across a collider structure $X \to Z \leftarrow Y$ and verify that conditioning on the collider $Z$ induces statistical dependence between marginally independent causes.


In [ ]:
# Exercise 8.3: Numerical Verification of Collider "Explaining Away"
# Model: X ~ Ber(0.5) (Rain), Y ~ Ber(0.5) (Sprinkler)
# Z = (X or Y with noise) (Wet Grass)
N_mc = 200000
np.random.seed(42)

X = np.random.binomial(1, 0.5, size=N_mc)
Y = np.random.binomial(1, 0.5, size=N_mc)
# Grass is wet if either Rain or Sprinkler, with small background noise
p_wet = 0.9 * (X | Y) + 0.05
Z = np.random.rand(N_mc) < p_wet

# 1. Marginal Independence: P(X=1, Y=1) vs P(X=1)*P(Y=1)
p_x1 = np.mean(X == 1)
p_y1 = np.mean(Y == 1)
p_x1_y1 = np.mean((X == 1) & (Y == 1))
cov_xy = np.cov(X, Y)[0, 1]

# 2. Conditional Dependence given Z=1 (Grass is Wet)
wet_idx = (Z == 1)
X_given_Z = X[wet_idx]
Y_given_Z = Y[wet_idx]

p_x1_given_Z = np.mean(X_given_Z == 1)
p_y1_given_Z = np.mean(Y_given_Z == 1)
p_x1_y1_given_Z = np.mean((X_given_Z == 1) & (Y_given_Z == 1))
cov_xy_given_z = np.cov(X_given_Z, Y_given_Z)[0, 1]

print("--- Directed Graphical Model: Collider Independence Verification ---")
print(f"Marginal: P(X=1) = {p_x1:.4f}, P(Y=1) = {p_y1:.4f}")
print(f"Marginal: P(X=1, Y=1) = {p_x1_y1:.4f} vs P(X)*P(Y) = {p_x1 * p_y1:.4f}")
print(f"Marginal Covariance Cov(X, Y) = {cov_xy:.5f} (Marginally Independent!)")
print("-" * 50)
print(f"Conditioned on Z=1 (Collider observed):")
print(f"P(X=1 | Z=1) = {p_x1_given_Z:.4f}, P(Y=1 | Z=1) = {p_y1_given_Z:.4f}")
print(f"Joint: P(X=1, Y=1 | Z=1) = {p_x1_y1_given_Z:.4f} vs Product = {p_x1_given_Z * p_y1_given_Z:.4f}")
print(f"Conditional Covariance Cov(X, Y | Z=1) = {cov_xy_given_z:.5f} (Dependent / Explaining Away!)")


---
## Exercise 8.4 — Analytical Bias-Variance Decomposition & Multiprocessing Simulation

### Problem Statement
In Section 8.3 of *Mathematics for Machine Learning*, the generalization error is decomposed into:
$$\mathbb{E}_{\mathcal{D}}\left[(y - \hat{f}_{\mathcal{D}}(x))^2\right] = \operatorname{Bias}^2(x) + \operatorname{Variance}(x) + \sigma^2$$

In this exercise:
We simulate $M = 1000$ independent training datasets ($N_{\text{train}} = 25$) drawn from $f(x) = \sin(\pi x)$ with Gaussian noise $\sigma = 0.3$. Using `simulate_bias_variance_chunk` parallelized via `ThreadPoolExecutor`, we evaluate polynomial models of degrees $d \in \{1, \dots, 10\}$ and plot the bias-variance tradeoff curve.


In [ ]:
# Exercise 8.4: Parallel Bias-Variance Simulation
degrees = list(range(1, 11))
n_trials = 1000
N_train = 25
noise_std = 0.3
x_test = np.linspace(-1.0, 1.0, 100)

tasks = [(deg, n_trials, N_train, noise_std, x_test, 42 + deg) for deg in degrees]

print(f"Distributing Bias-Variance simulation ({n_trials} Monte Carlo trials per polynomial degree)...")
with ThreadPoolExecutor() as executor:
    results = list(executor.map(simulate_bias_variance_chunk, tasks))

f_true = np.sin(np.pi * x_test)
bias_sq_list = []
variance_list = []
total_mse_list = []
all_predictions = {}

for deg, preds in sorted(results, key=lambda x: x[0]):
    all_predictions[deg] = preds
    f_bar = np.mean(preds, axis=0)
    pointwise_bias_sq = (f_true - f_bar)**2
    avg_bias_sq = np.mean(pointwise_bias_sq)
    pointwise_var = np.mean((preds - f_bar)**2, axis=0)
    avg_variance = np.mean(pointwise_var)
    avg_mse = avg_bias_sq + avg_variance + noise_std**2
    
    bias_sq_list.append(avg_bias_sq)
    variance_list.append(avg_variance)
    total_mse_list.append(avg_mse)
    print(f"Degree {deg:2d}: Bias^2 = {avg_bias_sq:.5f}, Variance = {avg_variance:.5f}, Total MSE = {avg_mse:.5f}")


In [ ]:
# Plot Bias-Variance Tradeoff Curve and Model Ensembles
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

ax1.plot(degrees, bias_sq_list, 'o-', color="royalblue", lw=2.5, label=r"$\mathrm{Bias}^2$ (Underfitting)")
ax1.plot(degrees, variance_list, 's-', color="crimson", lw=2.5, label=r"$\mathrm{Variance}$ (Overfitting)")
ax1.axhline(noise_std**2, color="gray", ls="--", lw=1.5, label=r"Irreducible Error $\sigma^2$")
ax1.plot(degrees, total_mse_list, 'D-', color="darkorchid", lw=2.5, label=r"Total Expected MSE")
optimal_deg = degrees[np.argmin(total_mse_list)]
ax1.axvline(optimal_deg, color="forestgreen", ls=":", lw=2, label=f"Optimal Complexity (Degree {optimal_deg})")
ax1.set_title("The Bias-Variance Tradeoff in Polynomial Regression", fontsize=12, fontweight="bold")
ax1.set_xlabel("Model Complexity (Polynomial Degree $d$)")
ax1.set_ylabel("Expected Error / Loss")
ax1.set_xticks(degrees)
ax1.set_ylim(0, 0.35)
ax1.legend()
ax1.grid(True, alpha=0.3)

deg_sample = [1, 3, 9]
colors = ["royalblue", "forestgreen", "crimson"]
for deg, col in zip(deg_sample, colors):
    preds = all_predictions[deg]
    for i in range(15):
        ax2.plot(x_test, preds[i], color=col, alpha=0.15, lw=1)
    ax2.plot(x_test, np.mean(preds, axis=0), color=col, lw=2.5, label=rf"Degree {deg} Mean $\bar{{f}}(x)$")

ax2.plot(x_test, f_true, 'k--', lw=2.5, label=r"Ground Truth $\sin(\pi x)$")
ax2.set_title("Model Realizations across Dataset Resamplings", fontsize=12, fontweight="bold")
ax2.set_xlabel("Input $x$")
ax2.set_ylabel("Prediction $f(x)$")
ax2.set_ylim(-1.6, 1.6)
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## Exercise 8.5 — Model Selection: Cross-Validation vs. Information Criteria (AIC & BIC)

### Problem Statement
In Section 8.6 of *Mathematics for Machine Learning*, model selection avoids overfitting by penalizing complexity:
1. **$K$-Fold Cross-Validation:** Approximates test risk empirically by partitioning data into $K$ folds.
2. **Akaike Information Criterion (AIC):** $\text{AIC} = 2k - 2\ln \hat{L}$
3. **Bayesian Information Criterion (BIC):** $\text{BIC} = k\ln N - 2\ln \hat{L}$

In this exercise:
Fit polynomial models $d \in \{1, \dots, 8\}$ to a noisy non-linear dataset ($N=50$) and compare model selection results between 5-fold cross-validation, AIC, and BIC.


In [ ]:
# Exercise 8.5: Model Selection Comparison: 5-Fold CV vs AIC vs BIC
np.random.seed(123)
N_ms = 50
x_ms = np.random.uniform(-1.5, 1.5, size=N_ms)
y_ms = 0.5 * x_ms**3 - x_ms + 0.3 * np.random.randn(N_ms)

candidate_degrees = list(range(1, 9))
cv_scores = []
aic_scores = []
bic_scores = []

# 5-fold CV splits
K_folds = 5
indices = np.arange(N_ms)
np.random.shuffle(indices)
folds = np.array_split(indices, K_folds)

for d in candidate_degrees:
    # 1. 5-Fold CV
    fold_errs = []
    for f in range(K_folds):
        val_idx = folds[f]
        tr_idx = np.setdiff1d(indices, val_idx)
        
        c = np.polyfit(x_ms[tr_idx], y_ms[tr_idx], deg=d)
        pred_val = np.polyval(c, x_ms[val_idx])
        fold_errs.append(np.mean((y_ms[val_idx] - pred_val)**2))
    cv_scores.append(np.mean(fold_errs))
    
    # 2. Full fit for AIC / BIC
    c_full = np.polyfit(x_ms, y_ms, deg=d)
    residuals = y_ms - np.polyval(c_full, x_ms)
    rss = np.sum(residuals**2)
    sigma2_hat = rss / N_ms
    ll_hat = -0.5 * N_ms * (np.log(2 * np.pi * sigma2_hat) + 1.0)
    
    # Parameters k = d + 1 (coeffs) + 1 (noise variance)
    k_params = (d + 1) + 1
    aic = 2 * k_params - 2 * ll_hat
    bic = k_params * np.log(N_ms) - 2 * ll_hat
    
    aic_scores.append(aic)
    bic_scores.append(bic)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(candidate_degrees, cv_scores, 'o-', color="darkorchid", lw=2.5, label="5-Fold Cross-Validation MSE")
opt_cv = candidate_degrees[np.argmin(cv_scores)]
ax1.axvline(opt_cv, color="forestgreen", ls=":", lw=2, label=f"Selected Degree via CV (d* = {opt_cv})")
ax1.set_title("5-Fold Cross-Validation Error", fontsize=12, fontweight="bold")
ax1.set_xlabel("Polynomial Degree $d$")
ax1.set_ylabel("Mean Squared Error")
ax1.set_xticks(candidate_degrees)
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(candidate_degrees, aic_scores, 's--', color="royalblue", lw=2, label="AIC Score")
ax2.plot(candidate_degrees, bic_scores, 'o-', color="crimson", lw=2, label="BIC Score")
opt_bic = candidate_degrees[np.argmin(bic_scores)]
ax2.axvline(opt_bic, color="forestgreen", ls=":", lw=2, label=f"Selected Degree via BIC (d* = {opt_bic})")
ax2.set_title("Information Criteria (AIC & BIC)", fontsize=12, fontweight="bold")
ax2.set_xlabel("Polynomial Degree $d$")
ax2.set_ylabel("Score (Lower is Better)")
ax2.set_xticks(candidate_degrees)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Optimal Model Selected: CV = Degree {opt_cv} | BIC = Degree {opt_bic} (True Cubic Ground Truth Recovered!)")


---
### Key Takeaways from Chapter 8
1. **ERM and Loss Selection:** Squared error ($L_2$) corresponds to Gaussian likelihood and is sensitive to outliers. Absolute error ($L_1$, Laplace) and Huber loss provide robust alternatives whose influence functions saturate for large residuals.
2. **Frequentist vs. Bayesian Estimation:** MLE maximizes data likelihood, while MAP incorporates prior domain knowledge as a regularizer. As $N \to \infty$, the data dominates the prior and $\boldsymbol{\theta}_{\text{MAP}} \to \boldsymbol{\theta}_{\text{ML}}$.
3. **Directed Graphical Models:** Conditional independence properties are determined by graph topology. Colliders ($X \to Z \leftarrow Y$) are marginally independent, but conditioning on the collider induces dependence ("explaining away").
4. **Bias-Variance Dilemma:** Underfitting results from high bias (insufficient capacity); overfitting results from high variance (excess capacity). Model selection via CV, AIC, and BIC objectively balances this trade-off to recover the optimal hypothesis complexity.
